In [ ]:
%pip install -e ../../../

In [ ]:
import pandas as pd

def demand_pattern_gen(x: int) -> str:
    return f"DP{x:02d}"

# We skip first row where is written: Hourly demand profiles (L/s)
profiles_df = pd.read_csv(
    '../raw-data/DemandCurves.csv',
    delimiter=';',
    header=1, 
    index_col=0,
)

# We rename the columns
profiles_df.columns = [demand_pattern_gen(i) for i in range(1, len(profiles_df.columns)+1)]

# The row that contained the old header (index 0) is automatically skipped when using header=1.
# The row that contains the type information is now the *first* data row (index 0 in the DataFrame).
# Let's extract the type row.
type_row = profiles_df.iloc[0, :]
profiles_df = profiles_df.drop(profiles_df.index[0])

profile_types = dict(zip(profiles_df.columns, type_row))

# rename the index as hours from Midnight of 1stJanuary of the year
n_hours = len(profiles_df)  # Should be 8760 for a non-leap year
profiles_df['hour_of_the_year'] = [f'year_hour-{i}' for i in range(n_hours)]
profiles_df.set_index('hour_of_the_year', inplace=True)

# Convert everything to float
profiles_df = profiles_df.map(lambda x: float(str(x).replace(',', '.')))

# Get the normalized profiles:
profiles_df = profiles_df / profiles_df.mean()

profiles = {}
for ptype, type_label in zip(['residential', 'business'], ['Res', 'Non-res']):
    selected_cols = [col for col in profiles_df.columns if profile_types[col] == type_label]

    df = profiles_df[selected_cols].T
    df.index.name = 'demand_pattern_id'

    df = df.round(5)

    profiles[ptype] = df.reset_index()

In [ ]:
from water_futures_battle.core.base_model import StaticProperties

dempatterns_sps = StaticProperties('water_demand_model-static_properties', dataframes=profiles)

dempatterns_sps.dump('..')